# SE3082 - Parallel Computing
## Lab Sheet 9 – Introduction to CUDA (Google Colab Edition)
**Student Name:** Heshanth T  
**Student ID:** IT24101633  
**Degree Program:** BSc (Hons) in Information Technology

---


### Exercise 1 – Set up your CUDA environment in Google Colab
Step 3: Check GPU is visible

In [1]:
!nvidia-smi

+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05     CUDA Version: 12.2     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /  70W  |       0MiB / 15360MiB  |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+------------

Steps 4 & 5: Check compiler version and compute capability

In [2]:
!nvcc --version
!nvidia-smi --query-gpu=name,compute_cap --format=csv

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0
name, compute_cap
Tesla T4, 7.5


Step 7: Smoke test – hello.cu

In [3]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main(void) {
    hello<<<1, 4>>>();          // 1 block of 4 threads
    cudaDeviceSynchronize();    // wait so the GPU's output is flushed
    return 0;
}


Writing hello.cu


In [4]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


---
### Exercise 2 – Simple calculation program from the lecture (Adding two numbers)

In [5]:
%%writefile ex2_add.cu
#include <cstdio>

// Kernel: this code runs on the device (the NVIDIA GPU)
__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

// host = CPU, device = GPU
int main(void) {
    int a, b, c;                // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Set up input values
    a = 2;
    b = 7;

    // 1. Copy the inputs from host to device
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // 2. Launch the add() kernel on the GPU: 1 block, 1 thread
    add<<<1, 1>>>(d_a, d_b, d_c);

    // Colab edition: report a failed launch instead of silently printing garbage
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // 3. Copy the result from device to host
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result is %d\n", c);

    // Cleanup
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex2_add.cu


In [6]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add

Result is 9


---
### Exercise 3 – Vector addition using 512 blocks and one thread

In [7]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch add() kernel on GPU with N blocks, 1 thread per block
    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex3_blocks.cu


In [8]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
...
58 + 40 = 98
35 + 86 = 121
67 + 6 = 73
Verification: 0 mismatches out of 512


---
### Exercise 4 – Vector addition using one block and 512 threads

In [9]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch addT() kernel on GPU with 1 block of N threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex4_threads.cu


In [10]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
...
511) 67 + 6 = 73
Verification: 0 mismatches out of 512


#### Exercise 4 Experiment – Find the hardware block thread limit (N = 2048)

In [11]:
%%writefile ex4_threads_limit.cu
#include <cstdio>
#include <cstdlib>

#define N 2048                  // Attempting 2048 threads in 1 block (exceeds limit 1024)

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch addT() kernel on GPU with 1 block of N threads (N = 2048)
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex4_threads_limit.cu


In [12]:
!nvcc -arch=sm_75 ex4_threads_limit.cu -o ex4_threads_limit && ./ex4_threads_limit

Kernel launch failed: invalid configuration argument
Verification: 2048 mismatches out of 2048


---
### Exercise 5 – Multiply two vectors using blocks and threads (10,000,000 elements)

In [13]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000              // 10 million elements
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: global index of this thread (use blockIdx, blockDim, threadIdx)
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: only compute if i is inside the array. Why is this needed?
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies of a, b, c
    int *d_a, *d_b, *d_c;       // device copies of a, b, c
    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate the input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: number of blocks needed to cover N elements (round UP)
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    // TODO 4: launch vecMul with <<<blocks, THREADS_PER_BLOCK>>>
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 5: copy the result d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 6: print the last 1000 results as "index) a x b = c"
    for (int i = N - 1000; i < N; i++) {
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);
    }

    // TODO 7: verify all N results on the CPU and print the number of mismatches
    int mismatches = 0;
    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i]) mismatches++;
    }
    printf("Verification: %d mismatches out of %d\n", mismatches, N);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex5_vecmul.cu


In [14]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
...
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000


---
### Exercise 6 – Multiply two 10,000 × 10,000 matrices element by element (2D grid)

In [15]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>

#define ROWS 10000
#define COLS 10000

// Element-wise product: C[r][c] = A[r][c] * B[r][c]  (NOT matrix multiplication)
__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    // TODO 1: row comes from the y dimension, col from the x dimension
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // TODO 2: only compute if BOTH row and col are inside the matrix
    // TODO 3: convert (row, col) into a flat index: row * cols + col
    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // host copies (flat arrays, row-major)
    int *d_a, *d_b, *d_c;       // device copies
    size_t count = (size_t)ROWS * COLS;
    size_t size  = count * sizeof(int);         // 400 MB per matrix

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 32 x 16 = 512 threads per block
    dim3 threadsPerBlock(32, 16);
    // TODO 4: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);
    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    // TODO 5: launch mulElem with <<<numBlocks, threadsPerBlock>>>
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // TODO 6: copy d_c back into c on the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 7: print C[9999][9000] ... C[9999][9999]  (the last 1000 elements)
    for (int col = 9000; col < COLS; col++) {
        size_t idx = (size_t)9999 * COLS + col;
        if (col == 9000 || col == 9001 || col == 9998 || col == 9999)
            printf("C[9999][%d]: %d x %d = %d\n", col, a[idx], b[idx], c[idx]);
    }

    // TODO 8: verify all elements on the CPU and print the number of mismatches
    size_t mismatches = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) mismatches++;
    }
    printf("Verification: %zu mismatches out of %zu\n", mismatches, count);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex6_matrix.cu


In [16]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix

Grid: 313 x 625 blocks
C[9999][9000]: 29 x 56 = 1624
C[9999][9001]: 84 x 19 = 1596
C[9999][9998]: 45 x 72 = 3240
C[9999][9999]: 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
